# PointFusion3D — Day 2: Seeing in 3D
### Reboot × Spatial AI Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_Day_2.ipynb)

**Instructor:** Dwayne Jones

Today you write your first two point-cloud functions. This notebook has only what you need for
Session 2:

| Section | What you do |
|---|---|
| Setup | Install Open3D and download the scan data (about a minute) |
| Part 1 | Load a 3D scan of the Stanford Bunny and see it as a NumPy array |
| Part 2 | **TODO 1:** `remove_outliers()`, a KD-tree outlier filter |
| NumPy warm-up | The five NumPy moves you'll need, plus an optional three-question check |
| Part 3 | **TODO:** `find_correspondences()`, nearest-neighbor search between two scans |
| Homework | **TODO:** `sample_points_on_mesh()`, turn a real phone scan of Elmo into a point cloud |

Run cells top to bottom. Cells marked **`# TODO`** are yours to complete; everything else is
provided. No GPU needed: Colab's default CPU runtime is all you need.

Next session continues in the full notebook,
[`PointFusion3D_student_FULL.ipynb`](https://github.com/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_FULL.ipynb), where these same functions become the
ICP alignment algorithm.

## Setup
Run this once per Colab session. Takes about 30-60 seconds. It installs Open3D and, if the data
files aren't already next to this notebook, downloads them from the course GitHub repo.

In [ ]:
!pip install -q open3d
import os
import subprocess

# Get the data files. In Colab the notebook starts with an empty folder, so clone the repo
# and work from inside it. If you cloned the repo yourself, the data is already here.
if not os.path.exists("data/scanA_bunny_partial1.ply"):
    if not os.path.isdir("3d-portfolio-starter"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/Dwaynekj/3d-portfolio-starter.git"], check=True)
    os.chdir("3d-portfolio-starter")
print("Data files:", sorted(os.listdir("data")))

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

print("Open3D version:", o3d.__version__)

## Part 1 — What *is* a point cloud?

A point cloud is just a big array of `(x, y, z)` coordinates — one row per point a sensor
measured on a surface. That's it. No mesh, no faces, just a "cloud" of 3D samples. A LiDAR
sensor, a depth camera, or a phone's photogrammetry app (like the KIRI Engine demo
from today's slides) all produce one of these as their raw output.

Our object is the classic **Stanford Bunny**. On the left is what the real object looks like as a
solid 3D model (a mesh: a surface built from triangles). On the right is what a 3D scanner actually
hands you: just points sampled from that surface, with no triangles connecting them.

![The Stanford Bunny as a mesh and as a point cloud](https://raw.githubusercontent.com/Dwaynekj/3d-portfolio-starter/main/assets/stanford_bunny_mesh_vs_points.png)

Let's load the scan on the right and look at it as a plain NumPy array before we do anything
fancy with it.

In [ ]:
# Given: small helper to plot a point cloud (or two) in 3D. You'll reuse this a lot.
def plot_clouds(clouds, colors, labels, title="", s=2):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    for pts, c, lab in zip(clouds, colors, labels):
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=c, s=s, alpha=0.6, label=lab)
    all_pts = np.vstack(clouds)
    center = all_pts.mean(axis=0)
    radius = (all_pts.max(axis=0) - all_pts.min(axis=0)).max() / 2 * 1.1
    ax.set_xlim(center[0]-radius, center[0]+radius)
    ax.set_ylim(center[1]-radius, center[1]+radius)
    ax.set_zlim(center[2]-radius, center[2]+radius)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title)
    ax.legend()
    plt.show()

scan_a = o3d.io.read_point_cloud("data/scanA_bunny_partial1.ply")
pts_a = np.asarray(scan_a.points)
print("scan_a is just a NumPy array under the hood:", type(pts_a), pts_a.shape)
print("first 3 points (x, y, z):\n", pts_a[:3])

plot_clouds([pts_a], ["#3b82f6"], ["Scan A"], title=f"Scan A - {len(pts_a)} points")

## Part 2 — Real data is messy: filtering outliers

Now load **Scan B** — a second, independent scan of the same bunny from a different angle. In
the real world this would be a second pass of a phone scan, or a second LiDAR sweep. It's been
deliberately roughed up the way real sensor data actually looks:

- it's **rotated and shifted** relative to Scan A (different viewpoint)
- it's **noisy** (sensor measurement error)
- it's **partial** (the sensor didn't see the whole object)
- it has **outlier junk points** (reflections, sensor glitches — every real scan has these)

Plot it and you'll see the problem immediately.

In [ ]:
scan_b_raw = o3d.io.read_point_cloud("data/scanB_bunny_partial2_raw.ply")
pts_b_raw = np.asarray(scan_b_raw.points)
plot_clouds([pts_a, pts_b_raw], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (raw)"],
            title="Two independent scans of the same object")

### TODO 1: write your own outlier filter

Before we can match these two scans up, we need to throw out the junk points in Scan B. The
idea: **a real surface point has other points nearby; an outlier sits alone.**

You're given a KD-tree (a data structure for fast nearest-neighbor lookups — think of it like a
structure that lets you ask "what's near this point?" without checking every other point one by
one). Your job:

1. For every point, find the distance to its `k` nearest neighbors (excluding itself).
2. Compute the **mean** of those `k` distances — this is the point's "local density" score. A
   point buried inside the surface will have small, consistent neighbor distances. An outlier
   sitting off on its own will have a much larger mean distance.
3. A point is kept if its mean neighbor distance is **below** `global_mean + std_ratio * global_std`
   (i.e. it isn't a statistical outlier compared to the rest of the cloud).

Return a boolean NumPy array (`True` = keep this point).

In [ ]:
def remove_outliers(points, k=20, std_ratio=2.0):
    # points: (N, 3) array
    # k: number of nearest neighbors to check per point
    # std_ratio: how many standard deviations above the mean counts as an outlier
    # returns: (N,) boolean array, True = keep the point
    tree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(points)))
    mean_dists = np.zeros(len(points))

    # TODO: for each point, use tree.search_knn_vector_3d(point, k+1) to find its k nearest
    # neighbors (the +1 accounts for the point finding itself at distance 0 -- skip index 0).
    # search_knn_vector_3d returns (num_found, indices, squared_distances).
    # Fill in mean_dists[i] with the mean of the sqrt of those k squared distances.
    for i, p in enumerate(points):
        raise NotImplementedError("compute mean_dists[i] here")

    global_mean = mean_dists.mean()
    global_std = mean_dists.std()
    keep_mask = mean_dists < (global_mean + std_ratio * global_std)
    return keep_mask

keep = remove_outliers(pts_b_raw)
pts_b_clean = pts_b_raw[keep]
print(f"outlier removal: {len(pts_b_raw)} -> {len(pts_b_clean)} points ({(~keep).sum()} removed)")
plot_clouds([pts_a, pts_b_clean], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (cleaned)"],
            title="After outlier removal")

## NumPy you'll need (10 min warm-up)

Every point cloud in this notebook is an `(N, 3)` NumPy array: one row per point, columns
`x, y, z`. The same five NumPy moves show up again and again: in Part 3, in the ICP loop, and in
the homework. Run the demo cell below and **guess each printed shape before you read it.**

| Move | What it does | Where you'll use it |
|---|---|---|
| `axis=` | `pts.mean(axis=0)` collapses the rows, leaving one value per column: the centroid, shape `(3,)` | ICP (centering a scan), homework |
| Broadcasting | `(N, 3) - (3,)` subtracts from every row. A per-point `(N,)` array needs `[:, None]` first | ICP, homework |
| Boolean masks | `pts[mask]` keeps the rows where `mask` is `True` | Part 2 (`pts_b_raw[keep]`), ICP |
| Index arrays | `pts[idx]` picks rows by position, in that order, repeats allowed | Part 3 → ICP (matched pairs), homework (triangle corners) |
| `rng.choice(n, size, p=...)` | Random picks with the probabilities you choose | Homework |

This cell only uses Scan A from Part 1, so it runs even if your TODO 1 isn't working yet.

In [ ]:
# Given: run it, and check each printed shape against your guess.
pts = pts_a                                    # Scan A from Part 1, an (N, 3) array
print("pts:", pts.shape)

# 1. axis=: axis=0 collapses the rows (one answer per column), axis=1 collapses the columns
centroid = pts.mean(axis=0)
print("centroid:", centroid.shape, centroid.round(3))

# 2. Broadcasting: (N, 3) minus (3,) subtracts the centroid from every row
centered = pts - centroid
dist_from_center = np.linalg.norm(centered, axis=1)    # one distance per point
print("centered:", centered.shape, " dist_from_center:", dist_from_center.shape)

# 3. Boolean mask: keep the points closer to the center than the median point
mask = dist_from_center < np.median(dist_from_center)
inner = pts[mask]
print("mask:", mask.shape, mask.dtype, "-> inner:", inner.shape)

# 4. Index array: pick rows by position (row 0 twice, then rows 5 and 2)
idx = np.array([0, 0, 5, 2])
print("pts[idx]:", pts[idx].shape)

# 5. A per-point number times a per-point xyz: (N,) must become (N, 1) first
w = np.linspace(0, 1, len(pts))                # one weight per point
try:
    w * pts                                    # (N,) * (N, 3): the shapes don't line up
except ValueError as e:
    print("w * pts fails:", e)
scaled = w[:, None] * pts                      # (N, 1) * (N, 3) -> (N, 3)
print("w[:, None]:", w[:, None].shape, "-> scaled:", scaled.shape)

# 6. Weighted random picks: p has one probability per choice and sums to 1
rng = np.random.default_rng(0)
picks = rng.choice(3, size=10_000, p=[0.7, 0.2, 0.1])
print("picks:", picks.shape, " share of each choice:", np.bincount(picks) / len(picks))

plot_clouds([pts[~mask], inner], ["#94a3b8", "#3b82f6"], ["outer half (~mask)", "inner half (mask)"],
            title="A boolean mask picks rows")

### Quick check (optional): three one-liners

Replace each `None` with **one line** of NumPy, no loops. Then run the check cell under it.
Anything still `None` is skipped, so the rest of the notebook runs either way. If you can do all
three, you have every NumPy move the homework needs.

In [ ]:
# Q1: the size of Scan A's bounding box: max minus min along each axis, shape (3,)
bbox_size = None

# Q2: the 100 points of Scan A farthest from its centroid, shape (100, 3)
#     hint: np.argsort(dist_from_center) gives point indices from nearest to farthest
far_pts = None

# Q3: pull each point toward the centroid by its own amount t (t = 0 stays put, t = 1 lands on
#     the centroid). The answer is pts + t * (centroid - pts), with t fixed up to broadcast.
t = np.random.default_rng(1).random(len(pts))   # (N,) one amount per point
pulled = None

In [ ]:
# Given: checks your three answers
def _check(name, val, expected_shape, ok):
    if val is None:
        print(f"{name}: skipped")
    elif np.shape(val) != expected_shape:
        print(f"{name}: shape is {np.shape(val)}, expected {expected_shape}")
    else:
        print(f"{name}: {'PASS' if ok(np.asarray(val)) else 'right shape, wrong values'}")

_check("Q1 bbox_size", bbox_size, (3,),
       lambda v: np.allclose(v, np.ptp(pts, axis=0)))
_check("Q2 far_pts", far_pts, (100, 3),
       lambda v: np.linalg.norm(v - centroid, axis=1).min() >= np.sort(dist_from_center)[-100] - 1e-12)
_check("Q3 pulled", pulled, pts.shape,
       lambda v: np.allclose(np.linalg.norm(v - centroid, axis=1), (1 - t) * dist_from_center))

## Part 3 — Nearest-neighbor correspondence search

To align two point clouds, we first need to guess **which point in Scan A corresponds to which
point in Scan B**. We don't have labels telling us this — we have to search for it.

The simplest, surprisingly effective approach: **for every point in A, its correspondence in B
is just its nearest neighbor in B.** Some of these guesses will be wrong (especially early on,
before the scans are aligned) — that's expected and we'll deal with it next session.

This is the "nearest-neighbor search" building block — the same core idea as a k-NN classifier
or a k-NN lookup in a database, just applied to 3D coordinates instead of feature vectors.

In [ ]:
def find_correspondences(src_points, tgt_points):
    # For every point in src_points, find the index of its nearest point in tgt_points.
    # src_points: (N, 3) array
    # tgt_points: (M, 3) array
    # returns:
    #   indices: (N,) int array -- indices[i] is the row in tgt_points closest to src_points[i]
    #   distances: (N,) float array -- the corresponding distance
    tgt_cloud = o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(tgt_points))
    tgt_tree = o3d.geometry.KDTreeFlann(tgt_cloud)

    indices = np.zeros(len(src_points), dtype=np.int64)
    distances = np.zeros(len(src_points))

    # TODO: for each point p in src_points, call tgt_tree.search_knn_vector_3d(p, 1) to get
    # the single nearest neighbor in tgt_points. It returns (num_found, [index], [sq_distance]).
    # Store the index in indices[i] and sqrt(sq_distance) in distances[i].
    for i, p in enumerate(src_points):
        raise NotImplementedError("fill in indices[i] and distances[i]")

    return indices, distances

# quick sanity check on a tiny toy example you can verify by eye
toy_src = np.array([[0., 0., 0.], [5., 5., 5.]])
toy_tgt = np.array([[0.1, 0., 0.], [10., 10., 10.], [5.2, 4.9, 5.0]])
idx, dist = find_correspondences(toy_src, toy_tgt)
print("matched indices:", idx, " (expected [0 2])")
print("distances:", dist)
assert list(idx) == [0, 2], "toy_src[0] should match toy_tgt[0], toy_src[1] should match toy_tgt[2]"
print("PASS: toy example matched correctly")

## ✅ That's Session 2

If Part 3 printed **PASS: toy example matched correctly**, you're done with today's in-class work.

**Before Session 3:** do the Homework below. It only needs Setup and Part 1 to have been run.

## Homework — Turn a phone scan into a point cloud (due before Session 3)

In Session 2 you saw KIRI Engine turn phone photos of an Elmo toy into a 3D model. That model is
a **mesh**: a surface made of triangles (about 100,000 of them), not a point cloud. Everything in
this notebook works on point clouds, so the first step with any real scan is to **sample points
from its surface**. You'll write that sampler yourself.

Run **Setup** and **Part 1** first (you need `np`, `o3d` and `plot_clouds`). Then run the cell below
to download the Elmo scan. It's kept out of the repo because it's 11 MB, and only this homework
needs it.

In [ ]:
# Given: download and unzip the Elmo scan from the repo's GitHub Release (runs once).
import urllib.request
import zipfile

ELMO_URL = "https://github.com/Dwaynekj/3d-portfolio-starter/releases/download/data-elmo-v1/elmo_scan.zip"
ELMO_DIR = "data/kiri/elmo_scan"
ELMO_OBJ = f"{ELMO_DIR}/3DModel.obj"

if not os.path.exists(ELMO_OBJ):
    os.makedirs(ELMO_DIR, exist_ok=True)
    urllib.request.urlretrieve(ELMO_URL, "data/kiri/elmo_scan.zip")
    with zipfile.ZipFile("data/kiri/elmo_scan.zip") as z:
        z.extractall(ELMO_DIR)  # the zip holds 3DModel.obj, .mtl and .jpg

o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)  # hide texture warnings
elmo_mesh = o3d.io.read_triangle_mesh(ELMO_OBJ)
elmo_vertices = np.asarray(elmo_mesh.vertices)    # (V, 3) corner positions
elmo_triangles = np.asarray(elmo_mesh.triangles)  # (T, 3) each row = 3 row-indices into elmo_vertices
print(f"Elmo mesh: {len(elmo_vertices)} vertices, {len(elmo_triangles)} triangles")

### HW TODO: sample points evenly from a mesh surface

Your function gets the mesh as two arrays and returns `n_points` random points spread **evenly over
the surface**. Write it with NumPy array operations, no Python `for` loops (vectorized ops are a
big part of why 3D code runs fast).

1. **Triangle areas.** For a triangle with corners `A, B, C`, the area is
   `0.5 * |cross(B - A, C - A)|`. Compute all of them at once with `np.cross` and
   `np.linalg.norm(..., axis=1)`.
2. **Pick triangles, weighted by area.** A triangle twice as big should get twice as many points.
   Use `rng.choice(num_triangles, size=n_points, p=...)`, where `p` is each triangle's share of the
   total area. If you pick triangles uniformly instead, small, detailed regions (like Elmo's face)
   get far too many points.
3. **Pick a random spot inside each chosen triangle.** Draw `r1 = np.sqrt(rng.random(n))` and
   `r2 = rng.random(n)`, then the point is `(1 - r1) * A + r1 * (1 - r2) * B + r1 * r2 * C`.
   This is the standard formula for a uniform point in a triangle. The `sqrt` matters: without it,
   points bunch up near corner `A`.

Hints: `A = vertices[triangles[:, 0]]` gives the first corner of *every* triangle at once, shape
`(T, 3)`. To use the `(n,)` arrays `r1` and `r2` with `(n, 3)` corner arrays, add an axis:
`r1[:, None]`.

In [ ]:
def sample_points_on_mesh(vertices, triangles, n_points, seed=0):
    # vertices: (V, 3) array of corner positions
    # triangles: (T, 3) int array; each row holds 3 row-indices into vertices
    # returns: (n_points, 3) array of points spread evenly over the surface
    rng = np.random.default_rng(seed)
    A = vertices[triangles[:, 0]]  # (T, 3) first corner of every triangle
    B = vertices[triangles[:, 1]]
    C = vertices[triangles[:, 2]]

    # TODO 1: areas = (T,) array of triangle areas
    # TODO 2: chosen = (n_points,) triangle indices, picked with probability proportional to area
    # TODO 3: r1, r2 and the uniform point inside each chosen triangle (formula above)
    raise NotImplementedError("sample points on the mesh surface")

elmo_pts = sample_points_on_mesh(elmo_vertices, elmo_triangles, n_points=20000)
print("sampled:", elmo_pts.shape)

### Check your sampler

Two automatic checks:

- **On the surface:** every sampled point should sit on Elmo's surface (within 0.0001 units). Points
  floating off the surface mean the step-3 formula is off.
- **Even coverage:** slice Elmo into 8 horizontal bands. Each band should get the same share of
  points as its share of surface area. A big mismatch means step 2 isn't weighting by area.

In [ ]:
assert elmo_pts.shape == (20000, 3), f"expected (20000, 3), got {elmo_pts.shape}"

# Check 1: distance from every sampled point to the mesh surface
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(elmo_mesh))
dist_to_surface = scene.compute_distance(o3d.core.Tensor(elmo_pts.astype(np.float32))).numpy()
on_surface = np.mean(dist_to_surface < 1e-4)
print(f"points on the surface: {on_surface:.1%}")

# Check 2: share of points vs share of surface area in 8 height bands
corners = elmo_vertices[elmo_triangles]  # (T, 3, 3)
tri_area = 0.5 * np.linalg.norm(np.cross(corners[:, 1] - corners[:, 0], corners[:, 2] - corners[:, 0]), axis=1)
tri_height = corners[:, :, 1].mean(axis=1)
bands = np.quantile(tri_height, np.linspace(0, 1, 9))
area_share = np.histogram(tri_height, bands, weights=tri_area)[0] / tri_area.sum()
point_share = np.histogram(elmo_pts[:, 1], bands)[0] / len(elmo_pts)
worst_gap = np.abs(area_share - point_share).max()
print(f"worst band mismatch: {worst_gap:.1%} (should be under 3%)")

assert on_surface > 0.999, "Some points aren't on the surface -- check step 3"
assert worst_gap < 0.03, "Coverage is uneven -- check that step 2 weights triangles by area"
print("PASS - your sampler spreads points evenly over Elmo's surface.")

plot_clouds([elmo_pts], ["#ef4444"], ["Elmo (sampled)"], title=f"Elmo as a point cloud - {len(elmo_pts)} points", s=1)

# Save it -- this is now the same kind of data as the bunny scans in Parts 1-6.
o3d.io.write_point_cloud("data/kiri/elmo_points.ply",
                         o3d.geometry.PointCloud(o3d.utility.Vector3dVector(elmo_pts)))
print("saved data/kiri/elmo_points.ply")